# Imports necesarios

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Cargar CSVs

In [2]:
nodes = pd.read_csv("../data/nodes.csv")
edges = pd.read_csv("../data/edges.csv")

print("Dimensión de nodes:", nodes.shape)
print("Dimensión de edges:", edges.shape)

display(nodes.head())
display(edges.head())

Dimensión de nodes: (156422, 6)
Dimensión de edges: (300386, 2)


,spotify_id,name,followers,popularity,genres,chart_hits
0,48WvrUGoijadXXCsGocwM4,Byklubben,1738.0,24,"['nordic house', 'russelater']",['no (3)']
1,4lDiJcOJ2GLCK6p9q5BgfK,Kontra K,1999676.0,72,"['christlicher rap', 'german hip hop']","['at (44)', 'de (111)', 'lu (22)', 'ch (31)', ..."
2,652XIvIBNGg3C0KIGEJWit,Maxim,34596.0,36,[],['de (1)']
3,3dXC1YPbnQPsfHPVkm1ipj,Christopher Martin,249233.0,52,"['dancehall', 'lovers rock', 'modern reggae', ...","['at (1)', 'de (1)']"
4,74terC9ol9zMo8rfzhSOiG,Jakob Hellman,21193.0,39,"['classic swedish pop', 'norrbotten indie', 's...",['se (6)']


,id_0,id_1
0,76M2Ekj8bG8W7X2nbx2CpF,7sfl4Xt5KmfyDs2T3SVSMK
1,0hk4xVujcyOr6USD95wcWb,7Do8se3ZoaVqUt3woqqSrD
2,38jpuy3yt3QIxQ8Fn1HTeJ,4csQIMQm6vI2A2SCVDuM2z
3,6PvcxssrQ0QaJVaBWHD07l,6UCQYrcJ6wab6gnQ89OJFh
4,2R1QrQqWuw3IjoP5dXRFjt,4mk1ScvOUkuQzzCZpT6bc0


# Crear has_chart_hits

In [6]:
nodes["has_chart_hits"] = nodes["chart_hits"].notna().astype(int)

print(nodes["has_chart_hits"].value_counts())
print(nodes["has_chart_hits"].value_counts(normalize=True))

has_chart_hits
0    136781
1     19641
Name: count, dtype: int64
has_chart_hits
0    0.874436
1    0.125564
Name: proportion, dtype: float64


In [7]:
display(nodes[["name", "chart_hits", "has_chart_hits"]].head(20))

,name,chart_hits,has_chart_hits
0,Byklubben,['no (3)'],1
1,Kontra K,"['at (44)', 'de (111)', 'lu (22)', 'ch (31)', ...",1
2,Maxim,['de (1)'],1
3,Christopher Martin,"['at (1)', 'de (1)']",1
4,Jakob Hellman,['se (6)'],1
5,Madh,['it (2)'],1
6,Juice,['se (4)'],1
7,Nehuda,['fr (1)'],1
8,VovaZiLvova,['ua (1)'],1
9,Nata Record,['do (1)'],1


In [8]:
print(nodes[["followers", "popularity"]].isna().sum())
display(nodes[["followers", "popularity"]].describe())

followers     4
popularity    0
dtype: int64


,followers,popularity
count,1.564180e+05,156422.000000
mean,8.622371e+04,21.157497
std,9.401001e+05,18.338290
min,0.000000e+00,0.000000
25%,2.400000e+01,4.000000
50%,3.630000e+02,18.000000
75%,6.258000e+03,34.000000
max,1.021569e+08,100.000000


In [9]:
nodes["followers"] = nodes["followers"].fillna(0)
nodes["popularity"] = nodes["popularity"].fillna(0)

In [10]:
print(nodes[["followers", "popularity"]].isna().sum())

followers     0
popularity    0
dtype: int64


In [11]:
G = nx.Graph()

G.add_nodes_from(nodes["spotify_id"])

G.add_edges_from(
    edges[["id_0", "id_1"]].itertuples(index=False, name=None)
)

print("Número de nodos:", G.number_of_nodes())
print("Número de aristas:", G.number_of_edges())

Número de nodos: 156326
Número de aristas: 300386


In [12]:
valid_nodes = set(nodes["spotify_id"])
G = G.subgraph(valid_nodes).copy()

print("Número de nodos finales:", G.number_of_nodes())
print("Número de aristas finales:", G.number_of_edges())

Número de nodos finales: 156320
Número de aristas finales: 300379


In [13]:
print("Es dirigido:", G.is_directed())
print("Número de nodos:", G.number_of_nodes())
print("Número de aristas:", G.number_of_edges())
print("Número de componentes conexas:", nx.number_connected_components(G))

Es dirigido: False
Número de nodos: 156320
Número de aristas: 300379
Número de componentes conexas: 4338


In [14]:
largest_cc = max(nx.connected_components(G), key=len)

print("Tamaño de la componente gigante:", len(largest_cc))
print("Porcentaje de nodos en componente gigante:", len(largest_cc) / G.number_of_nodes())

Tamaño de la componente gigante: 148380
Porcentaje de nodos en componente gigante: 0.9492067553735927


In [15]:
degrees = [degree for node, degree in G.degree()]

average_degree = sum(degrees) / len(degrees)

print("Grado medio:", average_degree)
print("Grado máximo:", max(degrees))
print("Grado mínimo:", min(degrees))

Grado medio: 3.843129477993859
Grado máximo: 1781
Grado mínimo: 0


In [16]:
degree = dict(G.degree())

nodes["degree"] = nodes["spotify_id"].map(degree).fillna(0)

display(nodes[["name", "degree"]].sort_values("degree", ascending=False).head(10))

,name,degree
12406,Johann Sebastian Bach,1781
18735,Traditional,1371
5609,Mc Gw,858
13370,MC MN,632
11577,Jean Sibelius,580
2654,Armin van Buuren,513
8030,Gucci Mane,509
13017,Steve Aoki,498
19434,Snoop Dogg,495
7956,Diplo,494


In [17]:
degree_centrality = nx.degree_centrality(G)

nodes["degree_centrality"] = nodes["spotify_id"].map(degree_centrality).fillna(0)

display(
    nodes[["name", "degree", "degree_centrality"]]
    .sort_values("degree_centrality", ascending=False)
    .head(10)
)

,name,degree,degree_centrality
12406,Johann Sebastian Bach,1781,0.011393
18735,Traditional,1371,0.008771
5609,Mc Gw,858,0.005489
13370,MC MN,632,0.004043
11577,Jean Sibelius,580,0.003710
2654,Armin van Buuren,513,0.003282
8030,Gucci Mane,509,0.003256
13017,Steve Aoki,498,0.003186
19434,Snoop Dogg,495,0.003167
7956,Diplo,494,0.003160


In [18]:
clustering = nx.clustering(G)

nodes["clustering"] = nodes["spotify_id"].map(clustering).fillna(0)

display(
    nodes[["name", "degree", "clustering"]]
    .sort_values("clustering", ascending=False)
    .head(10)
)

,name,degree,clustering
8245,3enaba,2,1.0
156359,Dorrough Music,2,1.0
156327,emma løv,2,1.0
9215,SLAY,2,1.0
9332,Klang Ruler,2,1.0
9727,LEHTISET,2,1.0
9865,BUHAJ KLAN,2,1.0
9952,Tromba,2,1.0
10140,Giusy & Elettra,2,1.0
10206,GoToGuy,2,1.0


In [19]:
pagerank = nx.pagerank(G, alpha=0.85)

nodes["pagerank"] = nodes["spotify_id"].map(pagerank).fillna(0)

display(
    nodes[["name", "degree", "pagerank"]]
    .sort_values("pagerank", ascending=False)
    .head(10)
)

,name,degree,pagerank
12406,Johann Sebastian Bach,1781,0.003733
18735,Traditional,1371,0.002852
11577,Jean Sibelius,580,0.001136
5609,Mc Gw,858,0.001031
13370,MC MN,632,0.000812
17437,הכוכב הבא,377,0.000787
4518,John Williams,415,0.000748
9595,A.R. Rahman,463,0.000670
2654,Armin van Buuren,513,0.000655
19434,Snoop Dogg,495,0.000615


## Resumen final de las estadísticas

In [20]:
display(
    nodes[
        [
            "degree",
            "degree_centrality",
            "clustering",
            "pagerank"
        ]
    ].describe()
)

,degree,degree_centrality,clustering,pagerank
count,156422.000000,156422.000000,156422.000000,1.564220e+05
mean,3.850718,0.000025,0.080838,6.408507e-06
std,14.331503,0.000092,0.237056,2.061650e-05
min,0.000000,0.000000,0.000000,9.754775e-07
25%,1.000000,0.000006,0.000000,2.314763e-06
50%,1.000000,0.000006,0.000000,2.990022e-06
75%,2.000000,0.000013,0.000000,4.200979e-06
max,1781.000000,0.011393,1.000000,3.732686e-03


## Calcular comunidades

In [22]:
component_dict = {}

for i, component in enumerate(nx.connected_components(G)):
    for node in component:
        component_dict[node] = i

nodes["community"] = nodes["spotify_id"].map(component_dict).fillna(-1).astype(int)

print("Número de comunidades:", nodes["community"].nunique())
print(nodes["community"].value_counts().head(10))

Número de comunidades: 4338
community
0       148473
1998        66
3722        37
2461        22
2294        21
3280        21
3412        21
3000        20
356         19
22          16
Name: count, dtype: int64


## Creación de features_final.csv

In [23]:
features = nodes[
    [
        "spotify_id",
        "name",
        "followers",
        "popularity",
        "degree",
        "degree_centrality",
        "clustering",
        "pagerank",
        "community",
        "has_chart_hits"
    ]
].copy()

display(features.head())
print(features.shape)

,spotify_id,name,followers,popularity,degree,degree_centrality,clustering,pagerank,community,has_chart_hits
0,48WvrUGoijadXXCsGocwM4,Byklubben,1738.0,24,2,0.000013,0.000000,0.000005,0,1
1,4lDiJcOJ2GLCK6p9q5BgfK,Kontra K,1999676.0,72,64,0.000409,0.067956,0.000073,0,1
2,652XIvIBNGg3C0KIGEJWit,Maxim,34596.0,36,8,0.000051,0.107143,0.000013,0,1
3,3dXC1YPbnQPsfHPVkm1ipj,Christopher Martin,249233.0,52,39,0.000249,0.037787,0.000049,0,1
4,74terC9ol9zMo8rfzhSOiG,Jakob Hellman,21193.0,39,2,0.000013,0.000000,0.000007,0,1


(156422, 10)


In [24]:
print(features.isna().sum())

spotify_id           0
name                 4
followers            0
popularity           0
degree               0
degree_centrality    0
clustering           0
pagerank             0
community            0
has_chart_hits       0
dtype: int64


In [25]:
features.to_csv("../data/features_final.csv", index=False)

In [26]:
test = pd.read_csv("../data/features_final.csv")

display(test.head())
print(test.shape)
print(test.isna().sum())

,spotify_id,name,followers,popularity,degree,degree_centrality,clustering,pagerank,community,has_chart_hits
0,48WvrUGoijadXXCsGocwM4,Byklubben,1738.0,24,2,0.000013,0.000000,0.000005,0,1
1,4lDiJcOJ2GLCK6p9q5BgfK,Kontra K,1999676.0,72,64,0.000409,0.067956,0.000073,0,1
2,652XIvIBNGg3C0KIGEJWit,Maxim,34596.0,36,8,0.000051,0.107143,0.000013,0,1
3,3dXC1YPbnQPsfHPVkm1ipj,Christopher Martin,249233.0,52,39,0.000249,0.037787,0.000049,0,1
4,74terC9ol9zMo8rfzhSOiG,Jakob Hellman,21193.0,39,2,0.000013,0.000000,0.000007,0,1


(156422, 10)
spotify_id           0
name                 4
followers            0
popularity           0
degree               0
degree_centrality    0
clustering           0
pagerank             0
community            0
has_chart_hits       0
dtype: int64


In [28]:
features["name"] = features["name"].fillna("Unknown")
features.to_csv("../data/features_final.csv", index=False)

test = pd.read_csv("../data/features_final.csv")

print(test.shape)
print(test.isna().sum())

(156422, 10)
spotify_id           0
name                 0
followers            0
popularity           0
degree               0
degree_centrality    0
clustering           0
pagerank             0
community            0
has_chart_hits       0
dtype: int64


In [29]:
print(test["has_chart_hits"].value_counts())
print(test["has_chart_hits"].value_counts(normalize=True))

has_chart_hits
0    136781
1     19641
Name: count, dtype: int64
has_chart_hits
0    0.874436
1    0.125564
Name: proportion, dtype: float64


In [30]:
display(test.describe())

,followers,popularity,degree,degree_centrality,clustering,pagerank,community,has_chart_hits
count,1.564220e+05,156422.000000,156422.000000,156422.000000,156422.000000,1.564220e+05,156422.000000,156422.000000
mean,8.622151e+04,21.157497,3.850718,0.000025,0.080838,6.408507e-06,108.880714,0.125564
std,9.400882e+05,18.338290,14.331503,0.000092,0.237056,2.061650e-05,548.197688,0.331359
min,0.000000e+00,0.000000,0.000000,0.000000,0.000000,9.754775e-07,0.000000,0.000000
25%,2.400000e+01,4.000000,1.000000,0.000006,0.000000,2.314763e-06,0.000000,0.000000
50%,3.630000e+02,18.000000,1.000000,0.000006,0.000000,2.990022e-06,0.000000,0.000000
75%,6.258000e+03,34.000000,2.000000,0.000013,0.000000,4.200979e-06,0.000000,0.000000
max,1.021569e+08,100.000000,1781.000000,0.011393,1.000000,3.732686e-03,4337.000000,1.000000
